# GraphFleet Auto-Prompting Guide

This notebook demonstrates how to use GraphFleet's auto-prompting features, following GraphRAG's approach. We'll cover:

1. Understanding prompt templates
2. Auto-prompt generation
3. Prompt tuning
4. Custom templates

For more details, see [GraphRAG Prompt Tuning Documentation](https://microsoft.github.io/graphrag/prompt_tuning/overview/)

In [ ]:
import os
from pathlib import Path
from graphfleet.core import GraphFleet
from graphfleet.prompting import PromptGenerator, PromptTuner

# Initialize project
project_dir = Path("./data")
graph_fleet = GraphFleet(project_dir)

## 1. Understanding Prompt Templates

GraphFleet uses GraphRAG's template system for generating prompts. Let's look at the default templates:

In [ ]:
# List available templates
templates = graph_fleet.list_templates()
print("Available Templates:")
for template in templates:
    print(f"- {template}")

# View a template
standard_template = graph_fleet.get_template("standard")
print("\nStandard Template:")
print(standard_template)

## 2. Auto-Prompt Generation

Now let's generate prompts automatically based on our indexed content:

In [ ]:
# Initialize prompt generator
generator = PromptGenerator(
    model="gpt-4",           # Model for generating prompts
    num_prompts=5,           # Number of prompts to generate
    temperature=0.7          # Creativity level
)

# Generate prompts
prompts = await generator.generate_prompts(
    template_type="standard",
    context_window=5         # Number of context chunks to consider
)

print("Generated Prompts:")
for i, prompt in enumerate(prompts, 1):
    print(f"\nPrompt {i}:")
    print(prompt)

## 3. Prompt Tuning

Let's tune our prompts using GraphRAG's tuning approach:

In [ ]:
# Initialize prompt tuner
tuner = PromptTuner(
    evaluation_model="gpt-4",    # Model for evaluating prompts
    num_iterations=3,            # Number of tuning iterations
    population_size=10           # Number of prompts in population
)

# Define evaluation metrics
metrics = [
    "relevance",      # Relevance to query
    "coherence",      # Response coherence
    "conciseness"     # Response conciseness
]

# Tune prompts
tuned_prompts = await tuner.tune_prompts(
    prompts=prompts,
    metrics=metrics,
    show_progress=True
)

print("\nTuned Prompts:")
for i, (prompt, score) in enumerate(tuned_prompts, 1):
    print(f"\nPrompt {i} (Score: {score:.2f}):")
    print(prompt)

## 4. Custom Templates

You can also create custom templates for specific use cases:

In [ ]:
# Define custom template
custom_template = """
You are an AI assistant helping with technical documentation.
Based on the following context:

{context}

Please answer the following question:
{query}

Provide your answer in this format:
1. Direct answer (1-2 sentences)
2. Supporting details from context
3. Any relevant caveats or limitations
"""

# Register custom template
graph_fleet.register_template(
    name="technical_docs",
    template=custom_template
)

# Generate prompts with custom template
custom_prompts = await generator.generate_prompts(
    template_type="technical_docs",
    context_window=3
)

print("Generated Custom Prompts:")
for i, prompt in enumerate(custom_prompts, 1):
    print(f"\nPrompt {i}:")
    print(prompt)

## 5. Save and Load Prompts

Finally, let's save our best prompts for future use:

In [ ]:
# Save prompts
prompts_dir = project_dir / "prompts"
prompts_dir.mkdir(exist_ok=True)

# Save standard prompts
graph_fleet.save_prompts(
    prompts=tuned_prompts,
    template_type="standard",
    path=prompts_dir / "standard_prompts.json"
)

# Save custom prompts
graph_fleet.save_prompts(
    prompts=custom_prompts,
    template_type="technical_docs",
    path=prompts_dir / "technical_prompts.json"
)

print("Prompts saved successfully!")